In [1]:
from lcpy.calculators.helpers import ListHolder, ExchangeHolder, ImpactCalculator, ImpactHandler
from lcpy.calculators.bw_int import mpLCAer
import os
from lcpy.hvs.hvs import create_dataframe_dict, save_dataframes_to_excel, plot_stacked_percentage_barchart_seaborn, create_name_dictionaries, create_dataframes_for_holistic_contribution_analysis
from lcpy.hvs.map_dicts import create_mapping, create_list_with_unique_activities
from lcpy.hvs.hvs import plot_stacked_percentage_bar_sub_processes, plot_stacked_percentage_bar_grid, make_characterized_inventory_dfs_simple_lca
from lcpy.calculators.env_calc import fast_calculator

# General configuration parameters

In [2]:
target_dir = "C:\\Users\\sgkousis\\Desktop\\Example_results\\Premise_LCA"
os.makedirs(target_dir, exist_ok=True)

In [3]:
methods_list = [
('TRACI v2.1', 'acidification', 'acidification potential (AP)'),
('TRACI v2.1', 'climate change', 'global warming potential (GWP100)'),
('TRACI v2.1', 'ecotoxicity: freshwater', 'ecotoxicity: freshwater'),
('TRACI v2.1', 'eutrophication', 'eutrophication potential'),
('TRACI v2.1', 'human toxicity: carcinogenic', 'human toxicity: carcinogenic'),
('TRACI v2.1', 'human toxicity: non-carcinogenic', 'human toxicity: non-carcinogenic'),
('TRACI v2.1', 'ozone depletion', 'ozone depletion potential (ODP)'),
('TRACI v2.1', 'particulate matter formation', 'particulate matter formation potential (PMFP)'),
('TRACI v2.1', 'photochemical oxidant formation', 'maximum incremental reactivity (MIR)'),
]

method_units_list = ['kg SO2-Eq',
 'kg CO2-Eq',
 'CTUe',
 'kg N-Eq',
 'CTUh',
 'CTUh',
 'kg CFC-11-Eq',
 'kg PM2.5-Eq',
 'kg O3-Eq',
]

Lcpy can use databases generated by the premise software (https://www.sciencedirect.com/science/article/pii/S136403212200226X). The user needs to generate databases using premise and load them to a bw2 project (note to use the premise-bw2 package).
The modelling then is exactly the same as in the other LCA examples. Note that in the database used by the user in Lcpy processes from different premise-generated databases can be loaded.

Below we provide a minimal example with random processes, to demonstrate how premise can be used with lcpy.



In [4]:
brightway_configuration_dictionary = {
    "path_to_brightway_project": "C:\\Users\\sgkousis\\Desktop\\bright\\.venv\\Lib\\site-packages",
    "bw_project": "premise_project2",
    "bw_database": "premise_db",
    "bw_biosphere": "biosphere3",
    "bw_ecoinvent": "remind_ssp2_base_2028"
}

In [5]:
methods_gp = methods_list[:]
impact_categories_names = ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']

In [6]:
impact_categories_units = method_units_list
scenario_names = ['one']

In [7]:
timeframe = 2 #operational lifetime after construction
time_step = 1
construction_years = 0

# Simple parametric model

# Example:
We consider three sub_processes: Nuclear power generation, RES power generation, and fossil power generation (only natural gas)

We navigate in the activity browser or brightway in the project defined in the brightway_configuration_dictionary above, and we derive the
keys that point to activities representing nuclear power generation from boiling or pressure water reactors, natural gas power generation from conventional or combined
cycle plants for standalone and co-generation plants, hydro power generation from pumped storage, deep geothermal power generation, onshore wind power generation.
We consider the UK as our geographical reference.

We are using ecoinvent processes `as is'. Nevertheless, one could create their own processes and assign their own keys to them.

In [8]:
keys_nuclear_power_generation = {
'Something' :  '1fe931141357cca720e7de5bdcd8f88a_copy1',
'Something else' : 'a4b5d0903d179e0af829e37e9c25e210_copy1',
'Something else else': '29572a17244959a18a4790f94e3320bb_copy1'
}
nuclear_exchanges_amounts = [
0.5,
0.98,
0.7
]

keys_fossil_power_generation = {
'This' :  '5b1ff2a1166a5dfa3f70f8615300f0db',
'That' : 'ea53e67a8db812c8db09922ef5aa73d2',
}
fossil_exchanges_amounts = [
2.1,
0.12
]

keys_res_power_generation = {
'One' :  '66d1cd82930e34a674c32d6408d466a8',
'Two' : 'ecc23c99e4998b586ea08bfffc634347',
}
res_exchanges_amounts = [
0.77775,
0.354,
]

And here we set the dictionaries for the main process

In [9]:
keys_total_power_generation = {
'Sp1': '',
'Sp2' : '',
'Sp3' : '',
}

mp_exchanges_amounts = [
0.4,
0.34,
0.98,
]

To perform the analysis the dictionaries with keys and the lists with the exchange amounts for eahc sub-process need to be buckled up in lists

In [10]:
key_list_sub_processes = [keys_nuclear_power_generation, keys_fossil_power_generation, keys_res_power_generation]
exchanges_list_sub_processes = [nuclear_exchanges_amounts, fossil_exchanges_amounts, res_exchanges_amounts]

Creates a dictionary with keys the keys of the first dictionary and values of the dictionaries included in the second list

In [11]:
mapping_names = create_mapping(keys_total_power_generation, key_list_sub_processes)

Creates a list with the names of each sub-sub-processes considered

In [12]:
unique_activities = create_list_with_unique_activities(key_list_sub_processes)

In [13]:
mapping_exchanges = create_mapping(keys_total_power_generation, exchanges_list_sub_processes)

Creates an instance that can run brightway. It uses 4 cpus, it considers the impact assessment methods provided, and runs everything in the bw project specified in the bw_config_dictionary. Then this bw environment needs to be set up.

In [14]:
my_lca = mpLCAer(4, methods_gp, brightway_configuration_dictionary)

In [15]:
my_lca.import_isolated_environment()

Calculates the unit impact, inventories, and characterized inventories, for each sub-sub-process in each sub-process.
The impact, inventory, and characterized inventory values are stored in dictionaries of the my_lca object.

In [16]:
my_lca.lca_calculations(mapping_names)

This notebook is mainly concerned with calculating and presenting the LCA results in terms of impact categories. To calculate the inventory and the impact caused by each inventory flow, the below lines of code can be run which are better explained in the fully-dynamic LCA example and the technical documentation. To perfrom this analysis per environmental flow a fast_calculator instance need to be loaded (again better explained in the next examples (MC/GSA/semi-dynamic/fully-dynamic).

The next nine cells can be omitted if only the actual impacts are to be explored without being concerned about the environmental flows.

In [17]:
my_lca.derive_technosphere_and_biosphere_dictionaries(mapping_names, "Main process")

OutsideTechnosphere: Can't find key 'acetylene production' (kilogram, RER, None) in product dictionary

In [18]:
my_calculator = fast_calculator()

In [19]:
my_calculator.sum_inventory_per_sub_sub_process(my_lca.unit_inventory)

In [20]:
my_calculator.sum_characterized_inventory_per_sub_sub_process(my_lca.unit_char_inventory)

In [21]:
my_calculator.emissions_calculation_simple_lca(mapping_exchanges, my_calculator.summed_unit_inventories)

In [22]:
my_calculator.characterized_inventory_calculation_simple_lca(mapping_exchanges, my_calculator.summed_unit_characterized_inventories)

In [23]:
my_calculator.emissions_calculation_total_simple_lca(mp_exchanges_amounts, my_calculator.inventory, "Main process" )

In [24]:
my_calculator.characterized_inventory_calculation_total_simple_lca(mp_exchanges_amounts, my_calculator.characterized_inventory, "Main process")

In [25]:
dictionary_with_dfs_per_environmental_flow_mp = make_characterized_inventory_dfs_simple_lca(my_calculator.total_char_inventory['Main process'],
                                        impact_categories_names, my_lca.biosphere_dict['Main process'],
                                        0)

KeyError: 'Main process'

In [26]:
dictionary_with_dfs_per_environmental_flow_elec_cons = make_characterized_inventory_dfs_simple_lca(my_calculator.characterized_inventory['Sp1'],
                                        impact_categories_names, my_lca.biosphere_dict['Main process'],
                                        0)

KeyError: 'Main process'

Here the parenthesis that explores the inventory flows ends and the analysis for the impact categories continues.

Creates instances to handle the exchange and impacts for the sub-processes

In [27]:
sp_exchange_manager = ExchangeHolder(methods_gp)
sp_impact_calculator = ImpactCalculator()

Creates dictionary mapping the sub-processes to the amount of exchange of each included sub-sub-process

In [28]:
sp_exchange_manager.create_exchange_arrays(mapping_exchanges)

Scales, for each sub-process, the unit impact of each sub-sub-process with each exchange amount --> Calculate unit impact of each sub-process

In [29]:
sp_impact_calculator.impact_calculation_simple(my_lca.unit_impacts, sp_exchange_manager.exchanges_dict)

Creates a handler instance

In [30]:
sp_results_handler = ImpactHandler(impact_categories_names)

Creates dataframes handling the impact results for each sub-process (for the unit impact of each sub-process). They are held in a dictionary of the sp_results_handler

In [31]:
names_dictionary = create_name_dictionaries(mapping_names, key_list_sub_processes)
sp_results_handler.create_dataframes(sp_impact_calculator.simple_impacts, names_dictionary)

We need now to scale the unit results for the sub-processes with the exchange amounts for the main process.
We create an impact handler and a list holding the name of the main process

In [32]:
mp_results_handler = ImpactHandler(impact_categories_names)

In [33]:
main_process_names = ['Electricity production']

Summarizes the unit impact of each sub-process included in the main process in a dictionary of the handler

In [34]:
mp_results_handler.calculate_total_unit_impact(sp_results_handler.total_impact_arrays, main_process_names[0])

Creates instances to handle the exchange and impacts for the sub-processes

In [35]:
mp_exchange_manager = ExchangeHolder(methods_gp)
mp_impact_calculator = ImpactCalculator()

Create the mapping dictionary between the name of the main process and the exchange amounts for each sub-process

In [36]:
mapping_exchanges_mp = {main_process_names[0]: mp_exchanges_amounts}

Also create a mapping names dictionary to help later in the visualization and storage

In [37]:
names_dictionary_graphite_main = {main_process_names[0]: list(names_dictionary.keys()) }

And now we follow the same logic to scale the unit impact of the sub-processes with the exchange amounts needed for the main process

In [38]:
mp_exchange_manager.create_exchange_arrays(mapping_exchanges_mp)

In [39]:
mp_impact_calculator.impact_calculation_simple(mp_results_handler.total_unit_impact, mp_exchange_manager.exchanges_dict)

At this point the impact result for the main process is calculated and stored at the mp_impact_calculator.simple_impacts, next there are a number of functions
to exploit these results, such as perform contribution analysis, put the results in dataframes, visualize them etc.

Create dataframes with the total impact and impact contribution per sub-process (attributes of the mp_results_handler)

In [40]:
mp_results_handler.create_dataframes(mp_impact_calculator.simple_impacts, names_dictionary_graphite_main)

Create list with the results sorted in the dataframes. Caution that we put the results for the sub-processes separately and for the main process separately.
These are contribution values

In [41]:
df_list_results = list(sp_results_handler.df_contributions.values())
df_list_results.append(mp_results_handler.df_contributions['Electricity production'])

In [42]:
sheet_names = list(keys_total_power_generation.keys()) + ['Total'] # Names of excel sheet for storage

Create a dictionary with the dataframes containing the dfs with the contribution analysis results

In [43]:
dict_with_contribution_dfs = create_dataframe_dict(df_list_results, sheet_names)

Create list containing dfs with the actual results for each sub-process and the main process and put them in a dictionary

In [44]:
list_with_absolute_results = list(sp_results_handler.df_impacts.values())
list_with_absolute_results.append(mp_results_handler.df_impacts['Electricity production'])
dict_with_absolute_results_df = create_dataframe_dict(list_with_absolute_results, sheet_names)

Create filepaths to store results (also can be set at the start of the notebook)

In [45]:
excel_file_path_contribution_results = os.path.join(target_dir, 'Electricity production.xlsx')
excel_file_path_absolute_results = os.path.join(target_dir, 'Totals_graphite.xlsx')

And save the results

In [46]:
save_dataframes_to_excel(dict_with_contribution_dfs, excel_file_path_contribution_results)
save_dataframes_to_excel(dict_with_absolute_results_df, excel_file_path_absolute_results)

Excel file 'C:\Users\sgkousis\Desktop\Example_results\Premise_LCA\Electricity production.xlsx' saved successfully.
Excel file 'C:\Users\sgkousis\Desktop\Example_results\Premise_LCA\Totals_graphite.xlsx' saved successfully.


Calculate the contribution to the impact of the main process per sub-sub-process

In [47]:
sp_results_handler.contribution_to_total_impact_per_sub_sub_processes(dict_with_absolute_results_df, dict_with_contribution_dfs, unique_activities, name= 'electricity_production', target_dir = target_dir)

Plot the contribution per sub-sub-process and store figure

In [48]:
figure_file_path = os.path.join(target_dir, f"sub_process_contributions_subplot.png")
plot_stacked_percentage_bar_grid(sp_results_handler.contribution_per_sub_sub_process, ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR'], figure_file_path, figsize=(15, 12), dpi=600, top_x=5, xlabel="Activities", ylabel="Contribution (%)")

Per impact category as desired

In [49]:
# temp_list = list(dict_with_contribution_dfs.keys())
#
# for cat in ['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']:
#     figure_file_path = os.path.join(target_dir, f"sub_process_contributions{cat}.png")
#     plot_stacked_percentage_bar_sub_processes(sp_results_handler.contribution_per_sub_sub_process, cat, figure_file_path, verbose = 'False')

Plot and store contribution barcharts per sub-process and for the main process

In [50]:
# temp_list = list(dict_with_contribution_dfs.keys())
# temp_list_2 = [s.replace(" ", "_") for s in temp_list]
# for path, key in zip(temp_list_2, temp_list):
#     figure_file_path = os.path.join(target_dir, f"{path}.png")
#     plot_stacked_percentage_barchart_seaborn(dict_with_contribution_dfs[key][['AP', 'GWP100', 'ECFW', 'EP', 'HTC', 'HTNC', 'ODP', 'PMFP', 'MIR']], figure_file_path, verbose='False', figsize = (5,3), dpi = 600, tab = 'dark')

The results go to a depth of 2 after the main-process (sub-process and sub-sub-process). The brightway functions can be used to make a contribution analysis
for each sub-sub-process. The mpLCAer class includes these functions for bw2, taken from the github bw tutorials. Below how to use them through the mpLCAer class
given the ecoinvent key of the sub-sub-process

In [51]:
contribution_analysis_of_sub_sub_process_processes = my_lca.contribution_analysis_in_technosphere(['baacfa40ab20a187d037b569b8a7fca2_copy1'], methods_gp)

IndexError: list index out of range

In [52]:
contribution_analysis_of_sub_sub_process_emissions = my_lca.contribution_analysis_in_biosphere(['baacfa40ab20a187d037b569b8a7fca2_copy1'], methods_gp)

IndexError: list index out of range

# The main process as a sub-process

The modelling to use an externally calculated process as a sub-process is the same with the simple LCA.
